# HistoCNN: Colorectal Histopathology Patch Classification
### Multi-Run Hyperparameter Tuning with MLflow on Google Colab (T4 GPU)

This notebook implements **HistoCNN**, a lightweight custom 4-block CNN (~308k parameters) trained from scratch on the **NCT-CRC-HE** dataset (9,000 balanced H&E patches across 9 colorectal tissue classes).

It runs a structured **MLflow hyperparameter tuning sweep** to benchmark learning rates, batch sizes, and learning rate scheduling (Constant vs Cosine Annealing) to maximize validation macro recall.

### Workflow:
1. **Runtime Setup**: Verify GPU acceleration ().
2. **Install & Imports**: Install , , , , and .
3. **Model & Data**: Define HistoCNN and stream the 9,000-patch balanced subset.
4. **MLflow Sweep**: Benchmark 4 candidate training runs logged to MLflow ().
5. **Leaderboard & Evaluation**: Display the sorted leaderboard and evaluate the winning model ().
6. **Save Weights**: Export clean PyTorch weights () and Base64 encoded payload.


In [ ]:
# Cell 1: Environment Setup
!pip install -q mlflow torch torchvision datasets pillow
import torch
print("CUDA available:", torch.cuda.is_available(),
      "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 2: Imports
import os
import copy
import random
import base64
import io
import zlib
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from datasets import load_dataset
import mlflow

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Cell 3: Define assignment architecture and evaluation functions

CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

def official_classes() -> list:
    return ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

def make_transforms(train: bool):
    if train:
        return T.Compose([
            T.Resize((64, 64)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
    else:
        return T.Compose([
            T.Resize((64, 64)),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

class _HFSubsetDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image = item["image"]
        label = item["label"]
        if image.mode != "RGB":
            image = image.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def build_model(num_classes: int = 9) -> torch.nn.Module:
    class HistoCNN(nn.Module):
        def __init__(self, n_classes):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),

                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),

                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),

                nn.Conv2d(128, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((2, 2)),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128 * 2 * 2, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, n_classes),
            )

        def forward(self, x):
            x = self.features(x)
            x = self.classifier(x)
            return x

    return HistoCNN(num_classes)

def predict(model: torch.nn.Module, loader, device: torch.device) -> torch.Tensor:
    model.eval()
    model.to(device)
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch
            images = images.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu())
    return torch.cat(all_preds).long()

def classification_report(y_true: torch.Tensor, y_pred: torch.Tensor) -> dict:
    num_classes = 9
    y_true = y_true.long()
    y_pred = y_pred.long()
    total = y_true.size(0)
    accuracy = (y_true == y_pred).sum().item() / total if total > 0 else 0.0
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(y_true, y_pred):
        confusion[t.item(), p.item()] += 1
    recalls = []
    for c in range(num_classes):
        true_c = (y_true == c).sum().item()
        if true_c > 0:
            tp = ((y_true == c) & (y_pred == c)).sum().item()
            recalls.append(tp / true_c)
    macro_recall = sum(recalls) / len(recalls) if recalls else 0.0
    return {
        "accuracy": float(accuracy),
        "macro_recall": float(macro_recall),
        "confusion": confusion,
    }

test_model = build_model(9)
param_count = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Built HistoCNN model: {param_count:,} trainable parameters (Budget: 25k - 500k)")


In [ ]:
# Cell 4: Load dataset and build balanced subset (1,000 per class -> 9,000 total)
PER_CLASS = 1000
NUM_CLASSES = 9

print("Loading dataset from Hugging Face (streaming mode)...")
ds = load_dataset("1aurent/NCT-CRC-HE", split="NCT_CRC_HE_100K", streaming=True)

class_buckets = {c: [] for c in range(NUM_CLASSES)}
for item in ds:
    label = item["label"]
    if len(class_buckets[label]) < PER_CLASS:
        class_buckets[label].append({"image": item["image"], "label": label})
    if all(len(v) >= PER_CLASS for v in class_buckets.values()):
        break

all_items = []
for c in range(NUM_CLASSES):
    all_items.extend(class_buckets[c])

random.seed(42)
random.shuffle(all_items)

split_idx = int(len(all_items) * 0.85)
train_items = all_items[:split_idx]
val_items = all_items[split_idx:]

print(f"Dataset ready: {len(train_items)} train samples, {len(val_items)} validation samples")


In [ ]:
# Cell 5: MLflow Hyperparameter Tuning Sweep
mlflow.set_experiment("came_assignment_02_tuning")

SWEEP_CONFIGS = [
    {
        "name": "run_baseline_lr1e3_ep12_bs32",
        "lr": 1e-3,
        "epochs": 12,
        "batch_size": 32,
        "weight_decay": 1e-4,
        "use_cosine_lr": False,
    },
    {
        "name": "run_cosine_lr1e3_ep16_bs32",
        "lr": 1e-3,
        "epochs": 16,
        "batch_size": 32,
        "weight_decay": 1e-4,
        "use_cosine_lr": True,
    },
    {
        "name": "run_cosine_lr7e4_ep18_bs32",
        "lr": 7e-4,
        "epochs": 18,
        "batch_size": 32,
        "weight_decay": 1e-4,
        "use_cosine_lr": True,
    },
    {
        "name": "run_bs64_lr12e3_ep16",
        "lr": 1.2e-3,
        "epochs": 16,
        "batch_size": 64,
        "weight_decay": 1e-4,
        "use_cosine_lr": True,
    },
]

val_labels_list = [item["label"] for item in val_items]
y_true_val = torch.tensor(val_labels_list, dtype=torch.long)

best_overall_macro_recall = -1.0
best_overall_state_dict = None
best_overall_config_name = None

for cfg in SWEEP_CONFIGS:
    run_name = cfg["name"]
    lr = cfg["lr"]
    epochs = cfg["epochs"]
    batch_size = cfg["batch_size"]
    weight_decay = cfg["weight_decay"]
    use_cosine = cfg["use_cosine_lr"]

    print(f"\n{'='*70}\nStarting MLflow Run: {run_name}")
    print(f"Parameters: lr={lr}, epochs={epochs}, batch_size={batch_size}, cosine_lr={use_cosine}\n{'='*70}")

    train_ds = _HFSubsetDataset(train_items, make_transforms(True))
    val_ds = _HFSubsetDataset(val_items, make_transforms(False))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "learning_rate": lr,
            "epochs": epochs,
            "batch_size": batch_size,
            "weight_decay": weight_decay,
            "scheduler": "CosineAnnealingLR" if use_cosine else "None",
            "seed": 42,
        })

        torch.manual_seed(42)
        model = build_model(9).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs) if use_cosine else None

        best_val_macro_recall = -1.0
        best_state = None

        for epoch in range(epochs):
            model.train()
            train_loss_sum, train_correct, train_total = 0.0, 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                logits = model(images)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

                bs = labels.size(0)
                train_loss_sum += loss.item() * bs
                train_correct += (logits.argmax(dim=1) == labels).sum().item()
                train_total += bs

            if scheduler:
                scheduler.step()

            train_loss = train_loss_sum / train_total
            train_acc = train_correct / train_total

            model.eval()
            val_loss_sum, val_correct, val_total = 0.0, 0, 0
            val_preds_list = []
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    logits = model(images)
                    loss = criterion(logits, labels)

                    bs = labels.size(0)
                    val_loss_sum += loss.item() * bs
                    val_correct += (logits.argmax(dim=1) == labels).sum().item()
                    val_total += bs
                    val_preds_list.append(logits.argmax(dim=1).cpu())

            val_loss = val_loss_sum / val_total
            val_acc = val_correct / val_total
            val_preds = torch.cat(val_preds_list)

            report = classification_report(y_true_val, val_preds)
            val_macro_recall = report["macro_recall"]

            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_acc", train_acc, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_acc", val_acc, step=epoch)
            mlflow.log_metric("val_macro_recall", val_macro_recall, step=epoch)

            if val_macro_recall > best_val_macro_recall:
                best_val_macro_recall = val_macro_recall
                best_state = copy.deepcopy(model.state_dict())
                star = " ★ best"
            else:
                star = ""

            print(f"Epoch {epoch:2d}/{epochs-1} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} MacroRecall: {val_macro_recall:.4f}{star}")

        mlflow.log_metric("final_best_val_macro_recall", best_val_macro_recall)
        print(f"\n--> Run '{run_name}' finished. Best Val Macro Recall: {best_val_macro_recall:.4f}")

        if best_val_macro_recall > best_overall_macro_recall:
            best_overall_macro_recall = best_val_macro_recall
            best_overall_state_dict = best_state
            best_overall_config_name = run_name


In [ ]:
# Cell 6: MLflow Experiment Leaderboard
runs = mlflow.search_runs(experiment_names=["came_assignment_02_tuning"])

summary_cols = [
    "tags.mlflow.runName",
    "params.learning_rate",
    "params.epochs",
    "params.batch_size",
    "params.scheduler",
    "metrics.final_best_val_macro_recall",
]
available_cols = [c for c in summary_cols if c in runs.columns]
summary_df = runs[available_cols].sort_values(
    by="metrics.final_best_val_macro_recall", ascending=False
).reset_index(drop=True)

print("═" * 70)
print("MLflow Hyperparameter Tuning Leaderboard (Sorted by Validation Macro Recall)")
print("═" * 70)
display(summary_df)

print(f"\n🏆 WINNING CONFIGURATION: {best_overall_config_name}")
print(f"🏆 HIGHEST VALIDATION MACRO RECALL: {best_overall_macro_recall:.4f}")


In [ ]:
# Cell 7: Detailed Evaluation of Winning Checkpoint
winning_model = build_model(9)
winning_model.load_state_dict(best_overall_state_dict)
winning_model.to(device)

eval_ds = _HFSubsetDataset(val_items, make_transforms(False))
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False)

final_preds = predict(winning_model, eval_loader, device)
final_report = classification_report(y_true_val, final_preds)

print("═" * 70)
print("Validation Report for Winning Model Checkpoint")
print("═" * 70)
print(f"Accuracy    : {final_report['accuracy']:.4f}")
print(f"Macro Recall: {final_report['macro_recall']:.4f}")
print("\nConfusion Matrix:")
print(final_report["confusion"])

if final_report["macro_recall"] >= 0.80:
    print(f"\n✓ macro_recall >= 0.80 — Target for 35/35 on Stage B REACHED!")
elif final_report["macro_recall"] >= 0.72:
    print(f"\n~ macro_recall >= 0.72 — Good (scores ~30/35).")
else:
    print(f"\n! macro_recall < 0.72 — Consider tuning learning rate or training more epochs.")


In [ ]:
# Step 8: Save Model Weights (.pth) and Base64 Payload
import os
import io
import zlib
import base64

os.makedirs("weights", exist_ok=True)
weights_path = "weights/histocnn_best.pth"

# Save clean FP32 PyTorch weights
clean_state = {k: v.detach().cpu().float() for k, v in winning_model.state_dict().items()}
torch.save(clean_state, weights_path)
file_size_mb = os.path.getsize(weights_path) / (1024 * 1024)
print(f"✓ Successfully saved PyTorch weights to: {weights_path} ({file_size_mb:.2f} MB)")

# Also encode to Base64 (FP16 + zlib level 9)
def encode_model_to_b64(model: torch.nn.Module) -> str:
    state = {k: v.detach().cpu().half() for k, v in model.state_dict().items()}
    buf = io.BytesIO()
    torch.save(state, buf)
    compressed = zlib.compress(buf.getvalue(), level=9)
    return base64.b64encode(compressed).decode("ascii")

b64_string = encode_model_to_b64(winning_model)
print(f"Winning Run Name   : {best_overall_config_name}")
print(f"Encoded Base64 size: {len(b64_string):,} characters")

# Optional: Download checkpoint to local machine
try:
    from google.colab import files
    files.download(weights_path)
except Exception:
    pass


In [ ]:
# Cell 9: Verify load_trained_model roundtrip
TRAINED_MODEL_B64 = b64_string

def load_trained_model(device):
    model = build_model(9)
    raw = zlib.decompress(base64.b64decode(TRAINED_MODEL_B64.strip()))
    state = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)
    state = {k: v.float() if v.is_floating_point() else v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

test_device = torch.device("cpu")
loaded_model = load_trained_model(test_device)
print("Successfully loaded model from Base64 string onto CPU!")

cpu_preds = predict(winning_model, eval_loader, torch.device("cpu"))
loaded_preds = predict(loaded_model, eval_loader, torch.device("cpu"))
match = torch.equal(cpu_preds, loaded_preds)
print(f"Roundtrip predictions match perfectly: {match}")
assert match, "Loaded model predictions do not match winning model!"
print("✓ Roundtrip verification PASSED! Ready for CMS.")
